# Re-optimized PSO results — analysis for Sections 5.1–5.2

Compares the 2019 PSO results (`../largeCases`, recorded costs from the seeds that selected each design, older simulator) with the re-optimized ones (`../PSO_results_rerun`, `PSO_optimization_v2`, costs of the final design **and** of full cross-training from 35 independent replications).

**Before using labels in the paper:** confirm the mapping of `holding_costs_variant` (0/1) to IND/HPB in `rerun_analysis.H_VARIANT_LABELS`.

Outputs: `tables/*.csv`, `tables/*.tex`, `figs/*.pdf`.

In [1]:
import os, json, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import rerun_analysis as ra
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 8})
BASE = os.environ.get("ANALYSIS_BASE", "..")            # folder that holds the three result folders
OLD, PSO, GA = (os.path.join(BASE, f) for f in ("largeCases", "PSO_results_rerun", "GA_results_rerun"))
os.makedirs("figs", exist_ok=True); os.makedirs("tables", exist_ok=True)

old, new = ra.load_folder(OLD, "old"), ra.load_folder(PSO, "new")
print(f"old: {len(old)} files | new: {len(new)} files")
both = old.merge(new, on="case", suffixes=("_old", "_new"))
print(f"paired cases: {len(both)}")

old: 128 files | new: 128 files
paired cases: 128


## 1. Checks on the new results

In [2]:
print("new designs that cannot carry their workload:", new.loc[~new.feasible, "case"].tolist() or "none")
print("new designs with a stock level of 490 or more:", new.loc[new["max stock"] >= 490, "case"].tolist() or "none")
print("cost components add up:", bool(np.allclose(new[["stock", "backorders", "skills cost", "technicians cost"]].sum(axis=1), new.total)))
print(f"runtime per case: median {new.runtime_h.median():.2f} h, max {new.runtime_h.max():.2f} h | evaluations requested: median {new.evaluations.median():.0f}")

new designs that cannot carry their workload: none
new designs with a stock level of 490 or more: none
cost components add up: True
runtime per case: median 1.92 h, max 2.58 h | evaluations requested: median 2700


## 2. The cases whose 2019 design was infeasible

In [3]:
rep = both[~both.feasible_old][["case", "N_old", "M_old", "rho_old", "total_old", "total_new", "total_ff_new", "saving_new"]]
rep.columns = ["case", "N", "M", "rho", "recorded 2019 (not a real cost)", "new total", "full cross-training", "new saving"]
print(f"{len(rep)} cases re-optimized from an infeasible design")
rep.to_csv("tables/pso_repaired_cases.csv", index=False)
rep.style.format({"recorded 2019 (not a real cost)": "{:,.0f}", "new total": "{:,.0f}", "full cross-training": "{:,.0f}", "new saving": "{:.1%}"})

17 cases re-optimized from an infeasible design


,case,N,M,rho,recorded 2019 (not a real cost),new total,full cross-training,new saving
0,1,10,5,0.650000,"600,840","605,066","1,020,182",40.7%
3,4,20,10,0.650000,"1,556,079","1,581,349","3,025,866",47.7%
9,10,20,5,0.650000,"882,357","902,563","1,639,529",44.9%
10,11,10,10,0.650000,"1,098,671","1,089,277","2,012,788",45.9%
22,23,10,10,0.800000,"1,050,125","1,295,987","2,506,815",48.3%
31,32,20,10,0.800000,"1,579,772","1,626,537","3,052,619",46.7%
34,35,20,5,0.800000,"875,761","988,310","2,682,579",63.2%
43,44,10,5,0.800000,"603,056","667,947","1,023,930",34.8%
49,50,10,10,0.800000,"1,010,842","1,092,773","1,709,868",36.1%
58,59,20,5,0.800000,"130,314","145,057","645,097",77.5%


## 3. New against recorded costs, cases that were feasible in 2019

The difference combines three things: the recorded cost was optimistic (selection bias), the old search wasted its last generation and could be misled by cached infeasible designs, and the simulator changed (validated as equivalent). Positive = the new cost is higher.

In [4]:
ok = both[both.feasible_old].copy(); ok["change"] = ok.total_new / ok.total_old - 1
print(f"{len(ok)} cases: change mean {ok.change.mean():+.2%}, median {ok.change.median():+.2%}, "
      f"quartiles {ok.change.quantile(.25):+.2%} / {ok.change.quantile(.75):+.2%}; new lower in {(ok.change < 0).mean():.0%} of cases")
same = sum(np.array_equal(np.sort(a, axis=0), np.sort(b, axis=0)) or sorted(map(tuple, a)) == sorted(map(tuple, b)) for a, b in zip(ok.X_old, ok.X_new))
print(f"identical design (up to the order of technicians): {same} of {len(ok)}")
ra.factor_summary(ok.rename(columns={c + "_old": c for c in ra.FACTORS}), "change").style.format({"mean": "{:+.2%}", "median": "{:+.2%}", "min": "{:+.2%}", "max": "{:+.2%}"})

111 cases: change mean +0.12%, median +0.04%, quartiles -0.96% / +1.04%; new lower in 49% of cases
identical design (up to the order of technicians): 0 of 111


,factor,level,cases,mean,median,min,max
0,N,10.000000,56,+0.15%,+0.35%,-9.75%,+7.84%
1,N,20.000000,55,+0.09%,-0.17%,-4.89%,+9.46%
2,M,5.000000,55,-0.10%,-0.07%,-3.99%,+4.65%
3,M,10.000000,56,+0.34%,+0.29%,-9.75%,+9.46%
4,rho,0.650000,57,+0.04%,+0.04%,-9.75%,+9.46%
5,rho,0.800000,54,+0.20%,+0.05%,-6.11%,+6.74%
6,hmin,1.000000,55,+0.00%,-0.06%,-9.75%,+9.31%
7,hmin,100.000000,56,+0.24%,+0.10%,-9.32%,+9.46%
8,h_variant,0.000000,54,+0.33%,+0.09%,-6.11%,+7.99%
9,h_variant,2.000000,57,-0.07%,+0.02%,-9.75%,+9.46%


## 4. Selection bias: search seeds against independent seeds (same simulator, same design)

In [5]:
new["bias"] = new.search_total / new.total - 1          # negative = the search's own seeds flatter the design
print(f"search cost vs independent cost: mean {new.bias.mean():+.2%}, median {new.bias.median():+.2%}, "
      f"lower in {(new.bias < 0).mean():.0%} of cases, largest {new.bias.min():+.2%}")
ra.factor_summary(new, "bias").style.format({"mean": "{:+.2%}", "median": "{:+.2%}", "min": "{:+.2%}", "max": "{:+.2%}"})

search cost vs independent cost: mean -0.26%, median -0.09%, lower in 70% of cases, largest -2.82%


,factor,level,cases,mean,median,min,max
0,N,10.000000,64,-0.37%,-0.17%,-2.82%,+1.09%
1,N,20.000000,64,-0.15%,-0.02%,-2.70%,+1.61%
2,M,5.000000,64,-0.30%,-0.10%,-2.70%,+1.61%
3,M,10.000000,64,-0.21%,-0.09%,-2.82%,+1.09%
4,rho,0.650000,64,-0.09%,-0.05%,-1.22%,+0.31%
5,rho,0.800000,64,-0.43%,-0.17%,-2.82%,+1.61%
6,hmin,1.000000,64,-0.22%,-0.10%,-2.65%,+0.55%
7,hmin,100.000000,64,-0.30%,-0.09%,-2.82%,+1.61%
8,h_variant,0.000000,64,-0.32%,-0.10%,-2.82%,+1.61%
9,h_variant,2.000000,64,-0.20%,-0.09%,-2.64%,+1.09%


## 5. Savings against full cross-training (Section 5.2)

In [6]:
s_new, s_old = new.saving, both.loc[both.feasible_old, "saving_old"]
print(f"new: mean {s_new.mean():.1%}, median {s_new.median():.1%}, range {s_new.min():.1%} to {s_new.max():.1%} ({len(s_new)} cases)")
print(f"2019 recorded, feasible cases only: mean {s_old.mean():.1%}, median {s_old.median():.1%}")
tab = ra.factor_summary(new, "saving")
tab["level"] = [ra.H_VARIANT_LABELS.get(l, l) if f == "h_variant" else l for f, l in zip(tab.factor, tab.level)]
tab["factor"] = tab.factor.map(lambda f: ra.FACTORS.get(f, f))
tab.to_csv("tables/pso_savings_by_factor.csv", index=False)
ra.latex_table(tab, "tables/pso_savings_by_factor.tex", "Cost savings of the optimized designs over full cross-training, by test-bed factor (35 independent replications).",
               "tab:savings_factor", {"mean": "{:.1%}", "median": "{:.1%}", "min": "{:.1%}", "max": "{:.1%}", "cases": "{:d}"})
tab.style.format({"mean": "{:.1%}", "median": "{:.1%}", "min": "{:.1%}", "max": "{:.1%}"})

new: mean 28.7%, median 31.0%, range 3.5% to 80.0% (128 cases)
2019 recorded, feasible cases only: mean 26.4%, median 28.0%


,factor,level,cases,mean,median,min,max
0,SKUs $N$,10.000000,64,27.1%,30.0%,3.5%,51.8%
1,SKUs $N$,20.000000,64,30.3%,35.4%,6.4%,80.0%
2,Technicians $M$,5.000000,64,27.6%,28.1%,3.5%,80.0%
3,Technicians $M$,10.000000,64,29.8%,34.3%,6.4%,51.8%
4,Utilization $\rho$,0.650000,64,27.4%,28.4%,3.5%,50.0%
5,Utilization $\rho$,0.800000,64,30.0%,33.3%,4.8%,80.0%
6,$h_{\min}$,1.000000,64,27.8%,28.1%,6.4%,77.5%
7,$h_{\min}$,100.000000,64,29.5%,32.7%,3.5%,80.0%
8,Holding cost/workload,IND,64,28.6%,29.3%,6.0%,80.0%
9,Holding cost/workload,HPB,64,28.8%,31.7%,3.5%,63.2%


In [7]:
# Figure: savings by factor, 2019 recorded (feasible cases) against re-optimized
comb = pd.concat([both.loc[both.feasible_old].rename(columns={c + "_old": c for c in ra.FACTORS}).assign(version="2019", saving=lambda x: x.saving_old),
                  new.assign(version="new")])[list(ra.FACTORS) + ["version", "saving"]]
facs = [f for f in ra.FACTORS if comb[f].nunique() > 1]
fig, axes = plt.subplots(1, max(len(facs), 1), figsize=(1.7 * max(len(facs), 1) + 0.8, 2.5), sharey=True, squeeze=False)
ra.boxplots_by_factor(axes[0], comb, "saving", ("version", ["2019", "new"]), ["white", "0.6"], ["2019", "new"], factors=facs)
axes[0, 0].set_ylabel("saving vs full cross-training (%)")
fig.legend([Patch(facecolor=c, edgecolor="black") for c in ["white", "0.6"]], ["2019 recorded (feasible cases)", "re-optimized, independent seeds"],
           loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.88)); fig.savefig("figs/pso_savings_by_factor.pdf"); plt.show()

## 6. Designs that cost more than full cross-training

The check behind the decision not to seed full cross-training: a negative saving means the swarm kept a design worse than one it could have had. Small negatives may be noise (the two costs come from 35 replications each); the optional cell re-evaluates them with more replications.

In [8]:
neg = new[new.saving < 0][["case", "N", "M", "rho", "total", "total_ff", "saving"]]
print(f"{len(neg)} of {len(new)} designs cost more than full cross-training")
neg.style.format({"total": "{:,.0f}", "total_ff": "{:,.0f}", "saving": "{:+.2%}"})

0 of 128 designs cost more than full cross-training


,case,N,M,rho,total,total_ff,saving


In [9]:
RUN_EXTRA = False            # True: re-evaluate those cases with 100 more replications (new seeds), a few seconds each
if RUN_EXTRA and len(neg):
    from simulation_codes import SimulationInterface_fast as SIf
    rows = []
    for c in neg.case:
        d = json.load(open(os.path.join(PSO, [f for f in os.listdir(PSO) if f.endswith(f"case_{c:04d}.json")][0])))
        lam, mu, h = (np.asarray(d[k], float) for k in ("failure_rates", "service_rates", "holding_costs")); b = d["penalty_cost"]
        X = np.asarray(d["best_assignment"]); f, cs = d["machine_cost"], d["skill_cost"]
        cd = SIf.simulation_optimization_run(lam, mu, h, b, X, replications=100, useDatabase=False, startSeed=5_000_000)[0] + f * (X.sum(1) > 0).sum() + cs * X.sum()
        cf = SIf.simulation_optimization_run(lam, mu, h, b, np.ones_like(X), replications=100, useDatabase=False, startSeed=5_000_000)[0] + f * X.shape[0] + cs * X.size
        rows.append({"case": c, "design (100 reps)": cd, "full CT (100 reps)": cf, "saving": 1 - cd / cf})
    display(pd.DataFrame(rows).style.format({"design (100 reps)": "{:,.0f}", "full CT (100 reps)": "{:,.0f}", "saving": "{:+.2%}"}))

## 7. Structure of the designs

In [10]:
cols = ["technicians", "skills", "skills per technician", "technicians per SKU", "SKUs with one technician"]
cmp = pd.DataFrame({"2019 (feasible cases)": both.loc[both.feasible_old, [c + "_old" for c in cols]].mean().values,
                    "new": new[cols].mean().values}, index=cols)
display(cmp.round(2))
st = new.groupby(["N", "M"])[cols].mean().round(2); st.to_csv("tables/pso_design_structure.csv"); st

,2019 (feasible cases),new
technicians,7.16,7.10
skills,42.32,41.64
skills per technician,5.96,5.91
technicians per SKU,2.82,2.77
SKUs with one technician,2.73,2.93


technicians  skills  skills per technician  technicians per SKU  SKUs with one technician
N  M                                                                                            
10 5          4.72   19.59                   4.18                 1.96                      3.81
   10         8.84   35.09                   3.97                 3.51                      0.53
20 5          5.00   42.12                   8.43                 2.11                      6.16
   10         9.84   69.75                   7.08                 3.49                      1.22

## 8. Convergence and run time

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(6.3, 2.4))
for (n, g), ls in zip(new.groupby("N"), ["-", "--"]):
    curves = [ra.normalized_curve(l) for l in g.log]; curves = [x for x in curves if x is not None]
    if not curves:
        axes[0].text(0.5, 0.5, 'no statistics in these logs', ha='center', transform=axes[0].transAxes); continue
    L = min(len(x) for x in curves); curves = np.array([x[:L] for x in curves]); gens = np.arange(L)
    axes[0].plot(gens, 100 * (np.nanmedian(curves, axis=0) - 1), ls, color="black", label=f"N = {n}")
    axes[0].fill_between(gens, 100 * (np.nanpercentile(curves, 25, axis=0) - 1), 100 * (np.nanpercentile(curves, 75, axis=0) - 1), color="0.85")
axes[0].set_xlabel("generation"); axes[0].set_ylabel("best cost above final (%)"); axes[0].legend(frameon=False)
axes[1].hist(new.runtime_h, bins=20, color="0.6", edgecolor="black"); axes[1].set_xlabel("run time per case (hours, one core)"); axes[1].set_ylabel("cases")
fig.tight_layout(); fig.savefig("figs/pso_convergence_runtime.pdf"); plt.show()

## 9. Per-case table

In [12]:
keep = ["case"] + list(ra.FACTORS) + ["total", "total_ff", "saving", "search_total", "bias", "technicians", "skills", "runtime_h", "evaluations"]
out = new[keep].merge(old[["case", "total", "total_ff", "feasible"]].rename(columns={"total": "total_2019", "total_ff": "total_ff_2019", "feasible": "feasible_2019"}), on="case")
out.to_csv("tables/pso_per_case.csv", index=False); print("tables/pso_per_case.csv:", len(out), "rows")

tables/pso_per_case.csv: 128 rows


## 10. Load of the busiest technician

Designs whose busiest technician works above 0.95 of capacity have the noisiest costs; their precision is in section 11.

In [13]:
new["max load"] = [np.asarray(json.load(open(os.path.join(PSO, f"fullRangeResultsPSO_new_2019_case_{c:04d}.json")))["utilizationRatesAverage"]).sum(axis=1).max() for c in new.case]
print(f"busiest technician: median {new['max load'].median():.3f}, max {new['max load'].max():.3f}; above 0.90 in {(new['max load'] > 0.90).sum()}, above 0.95 in {(new['max load'] > 0.95).sum()} designs")
new.loc[new["max load"] > 0.95, ["case", "N", "M", "rho", "max load", "bias"]].round(3)

busiest technician: median 0.893, max 0.993; above 0.90 in 56, above 0.95 in 19 designs


,case,N,M,rho,max load,bias
6,7,10,10,0.80,0.979,-0.001
10,11,10,10,0.65,0.967,-0.003
22,23,10,10,0.80,0.963,-0.001
26,27,10,10,0.65,0.952,0.003
30,31,10,10,0.80,0.973,-0.010
33,34,10,10,0.80,0.966,-0.002
40,41,20,10,0.80,0.959,0.000
41,42,10,10,0.80,0.963,-0.026
45,46,10,10,0.65,0.980,-0.006
58,59,20,5,0.80,0.972,-0.022


## 11. Precision of the reported costs (Section 4.3, Appendix B.4)

`tables/pso_cost_precision.csv` is written by `python3 cost_precision.py ../PSO_results_rerun tables/pso_cost_precision.csv` (about 15 minutes on one core): 95% half-widths from the 35 independent replications, at the reported stock levels.

In [14]:
prec = pd.read_csv("tables/pso_cost_precision.csv")
print(f"recomputed means equal the reported totals: {np.allclose(prec.design_mean, prec.design_reported, rtol=1e-9)}")
for c, lab in [("design_hw_rel", "final design, of the total cost"), ("design_hw_rel_variable", "final design, of stock + backorder cost"), ("ff_hw_rel", "full cross-training, of the total cost")]:
    print(f"{lab}: mean {prec[c].mean():.2%}, median {prec[c].median():.2%}, max {prec[c].max():.2%}")
hi = prec.merge(new[["case", "max load"]]).query("`max load` > 0.95")
print(f"designs above 0.95: half-width at most {hi.design_hw_rel.max():.2%}")

recomputed means equal the reported totals: True
final design, of the total cost: mean 0.49%, median 0.31%, max 3.11%
final design, of stock + backorder cost: mean 3.52%, median 2.48%, max 21.92%
full cross-training, of the total cost: mean 1.59%, median 0.34%, max 22.07%
designs above 0.95: half-width at most 2.11%
